# COMP5329 — Deep Learning

**Tutorial 10 — Deep Reinforcement Learning**

**Semester 1, 2026**

### Learning Objectives
By the end of this tutorial you will be able to:
1. Formulate a sequential decision problem as a **Markov Decision Process** and write the Bellman optimality equation for $Q^*$ from memory.
2. Explain why tabular Q-learning fails in large state spaces and motivate the move to neural function approximation.
3. **Implement** a Deep Q-Network from scratch — `QNetwork`, `ReplayBuffer`, and the TD-target loss — and articulate why the **target network** and the **experience replay buffer** are each individually necessary for stability.
4. Derive the **REINFORCE** policy-gradient estimator, identify its variance problem, and motivate advantages + trust-region updates.
5. **Implement** the **PPO clipped surrogate objective** and read the canonical two-panel clipping diagram for positive vs. negative advantage.
6. Answer exam-style short-answer questions on the DQN target network, PPO clipping behaviour, and the on-policy / off-policy distinction between the two families.

### Topic Coverage

Week 10 covers **Deep Reinforcement Learning**. The full topic list (see `Week10_Self_Study.ipynb`) is:

- ✅ **MDP framework** — states, actions, rewards, transitions, discount $\gamma$ *(tutorial)*
- ✅ **Bellman optimality equation** for $Q^*$ and tabular Q-learning update *(tutorial, markdown only)*
- ✅ **Deep Q-Networks (DQN)** — neural Q-function, experience replay, target network, end-to-end custom implementation *(tutorial)*
- ✅ **Policy gradient & REINFORCE** — log-derivative trick, variance problem, advantages *(tutorial)*
- ✅ **Proximal Policy Optimization (PPO)** — clipped surrogate objective, end-to-end custom implementation *(tutorial)*
- 📖 **Historical breakthroughs** — Atari, AlphaGo, StarCraft II *(self-study)*
- 📖 **RLHF full pipeline** — reward model training, KL-regularised PPO on LLMs *(self-study)*
- 📖 **DPO, SLiC-HF, GRPO** — PPO-family variants for preference optimisation *(self-study)*

Due to time constraints, the tutorial concentrates on **one value-based method (DQN)** and **one policy-gradient method (PPO)**. The LLM-alignment family (DPO, SLiC-HF, GRPO) builds directly on PPO and is covered in depth in the self-study notebook.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Review

> **Goal.** By the end of Part A you should be able to (a) write the Bellman optimality equation for $Q^*$ from memory, (b) explain why tabular methods break at scale and how DQN fixes it, and (c) explain why a vanilla policy-gradient estimator has a variance problem and how PPO bounds the update.
>
> Part A is **narrative only** — no new code is introduced here. All implementation happens in Part B.

---
## §1 — The MDP Framework

Reinforcement learning differs from supervised learning fundamentally: there are no labelled (input, output) pairs. Instead, an **agent** interacts with an **environment** by taking **actions**, observing **state transitions**, and receiving **rewards**.

A **Markov Decision Process** is a 5-tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

| Symbol | Name | Description |
|---|---|---|
| $\mathcal{S}$ | State space | All situations the agent can be in |
| $\mathcal{A}$ | Action space | All actions the agent can take |
| $P(s' \mid s, a)$ | Transition kernel | Probability of reaching $s'$ from $s$ after action $a$ |
| $R(s, a, s')$ | Reward function | Immediate reward for the transition |
| $\gamma \in [0, 1)$ | Discount factor | Trade-off between immediate and future reward |

The **Markov property**: the next state depends only on the current state and action, not on the history. A **policy** $\pi(a \mid s)$ maps states to a distribution over actions; the goal is to find $\pi^*$ that maximises the expected discounted return $\mathbb{E}_\pi\!\bigl[\sum_{t=0}^\infty \gamma^t R_t\bigr]$.

---
## §2 — Value Functions and the Bellman Optimality Equation

The **action-value function** under policy $\pi$ is
$$Q^\pi(s, a) \;=\; \mathbb{E}_\pi\!\left[\sum_{t=0}^\infty \gamma^t R_t \,\Big|\, S_0 = s,\; A_0 = a\right].$$

The optimal policy satisfies $V^*(s) = \max_a Q^*(s, a)$ and $\pi^*(s) = \arg\max_a Q^*(s, a)$. The function $Q^*$ is the unique fixed point of the **Bellman optimality equation**:

$$\boxed{\;Q^*(s, a) \;=\; \mathbb{E}_{s' \sim P(\cdot \mid s, a)}\!\Bigl[\,R(s, a, s') \;+\; \gamma \max_{a'} Q^*(s', a')\Bigr].\;}$$

This is the mathematical object every value-based RL algorithm tries to estimate. It is recursive: the value of taking action $a$ in state $s$ is the immediate reward plus the discounted best-case value of the next state.

### Tabular Q-learning (markdown only — no code in the tutorial)

If $\mathcal{S}$ and $\mathcal{A}$ are small enough to enumerate, we can store $Q$ in a table $|\mathcal{S}| \times |\mathcal{A}|$ and update it with the classic one-step TD rule after each transition $(s, a, r, s')$:

$$Q(s, a) \;\leftarrow\; Q(s, a) \;+\; \alpha\,\Bigl[\underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD target}} \;-\; Q(s, a)\Bigr].$$

The bracket is the **TD error** — the one-step residual of the Bellman equation, evaluated on a *sampled* transition instead of an expectation. With $\varepsilon$-greedy exploration and decaying $\alpha$, tabular Q-learning is guaranteed to converge to $Q^*$ for finite MDPs. You will *not* implement this in Part B — we go straight to the neural version.

---
## §3 — Why Tabular Q-Learning Breaks: The Move to DQN

Tabular Q-learning stores one scalar per $(s, a)$ pair. The moment the state space is continuous or combinatorially huge, the table is infeasible:

| Domain | $|\mathcal{S}|$ | Tabular? |
|---|---|---|
| $5\!\times\!5$ grid-world | 25 | ✅ trivial |
| Atari (84×84 grey pixels) | $256^{84 \times 84}$ | ❌ impossible |
| Robot joint angles | continuous | ❌ impossible |
| Board positions in Go | $\sim 10^{170}$ | ❌ impossible |

**Solution (Mnih et al., 2013/2015):** replace the table with a **neural network** $Q_\theta(s, \cdot)$ that maps a state to a vector of Q-values, one per action. Train it to minimise the squared Bellman residual on sampled transitions:

$$\mathcal{L}(\theta) \;=\; \mathbb{E}_{(s, a, r, s') \sim \mathcal{D}}\!\Bigl[\bigl(\,\underbrace{r + \gamma \max_{a'} Q_{\theta^-}(s', a')}_{\text{TD target}} \,-\, Q_\theta(s, a)\bigr)^2\Bigr].$$

Two innovations make this work in practice — without either, training diverges:

1. **Experience replay.** Transitions are pushed into a FIFO buffer $\mathcal{D}$; gradient updates sample random mini-batches instead of the most recent trajectory. This **breaks the temporal correlation** between consecutive samples (which violates the i.i.d. assumption of SGD) and **reuses each transition multiple times** (sample efficiency).
2. **Target network.** The TD target uses a *frozen copy* $\theta^-$ of the parameters, synchronised every $C$ steps (or via slow Polyak averaging). This **decorrelates the target from the current weights** — without it, every gradient step simultaneously moves the prediction *and* the target it is chasing, turning the update into a moving-target chase and breaking the contraction argument that guarantees convergence.

Part B Task B1/B2/B3 implements all three pieces: `QNetwork`, `ReplayBuffer`, and the `dqn_loss` TD-target computation.

---
## §4 — From Values to Policies: REINFORCE and its Variance Problem

Value-based methods (Q-learning, DQN) learn $Q$ and then act greedily. **Policy-gradient** methods instead parameterise the policy directly, $\pi_\theta(a \mid s)$, and ascend the expected return

$$J(\theta) \;=\; \mathbb{E}_{\tau \sim \pi_\theta}\!\Bigl[\sum_{t=0}^{T} \gamma^t R_t\Bigr].$$

Using the **log-derivative trick**, the gradient can be written as an expectation that we can sample:

$$\nabla_\theta J(\theta) \;=\; \mathbb{E}_{\tau \sim \pi_\theta}\!\Bigl[\sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot G_t\Bigr], \qquad G_t = \sum_{k \ge t} \gamma^{k-t} R_k.$$

This is the **REINFORCE** estimator (Williams, 1992). It is simple, on-policy, and unbiased — but notoriously **high variance**, because $G_t$ is a full Monte-Carlo return that can fluctuate wildly between trajectories. The standard fix is to subtract a state-dependent **baseline** $b(s_t)$ (typically a learned value function $V_\phi(s_t)$), leaving the **advantage** $\hat A_t = G_t - b(s_t)$ — same gradient in expectation, much smaller variance.

Even with advantages, there is a second problem. The gradient only tells us the *direction* of improvement, not a safe *step size*. Taking a step that is too large can push $\pi_\theta$ into a region where the collected data is no longer representative, and the policy collapses. This is the **trust-region** problem, and the next section's method — **PPO** — is its pragmatic answer.

---
## §5 — PPO: A Simple Trust-Region Fix

**Proximal Policy Optimization** (Schulman et al., 2017) reuses a batch of trajectories collected under an old policy $\pi_{\theta_\text{old}}$ for several gradient steps, but **bounds the per-step change** so the data never becomes stale. Define the **importance ratio**

$$r_t(\theta) \;=\; \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)} \;=\; \exp\!\bigl(\log \pi_\theta - \log \pi_{\theta_\text{old}}\bigr).$$

At the start of an epoch $r_t = 1$; as $\theta$ drifts away from $\theta_\text{old}$, the ratio moves away from 1. PPO's **clipped surrogate objective** is

$$\boxed{\;\mathcal{L}^{\text{CLIP}}(\theta) \;=\; \mathbb{E}_t\!\Bigl[\min\!\bigl(r_t(\theta)\,\hat A_t,\; \text{clip}(r_t(\theta),\,1-\varepsilon,\,1+\varepsilon)\,\hat A_t\bigr)\Bigr].\;}$$

**Why the min of two terms?** The clip alone would be *one-sided* optimism: if the ratio runs off to 100 with $\hat A_t > 0$, clipping it to $1+\varepsilon$ still yields a *positive* objective to maximise, so nothing stops the update. Taking the `min` of the clipped and unclipped terms means that once you leave the trust region on the "good" side, the gradient goes to zero — the update is **pessimistic outside the trust region** and there is no incentive to drift further. For $\hat A_t < 0$, the roles swap: the objective is bounded below at $1-\varepsilon$. You will reproduce this two-panel diagram in Part B.

### DPO, SLiC-HF, GRPO (one-paragraph mention, self-study)

Modern LLM alignment uses **PPO-family** variants that exploit the specific structure of preference data. **DPO** rewrites the KL-regularised RLHF objective in closed form and removes the reward model and the value critic, leaving a purely supervised classification-style loss on preference pairs. **SLiC-HF** replaces the RL loop entirely with a max-margin ranking loss. **GRPO** keeps PPO's clipped ratio but removes the value critic by computing advantages as *group-relative* normalised rewards within a batch of responses to the same prompt. All three are derivatives of the clipped surrogate you will implement in Task B5 — see `Week10_Self_Study.ipynb` for full derivations.

---
## §6 — Tiny Illustrative Plot: Discounted Return on a 4-State Chain

Before Part B, a quick sanity check on the Bellman equation. Consider a 4-state chain with a single "right" action that always succeeds, reward $+1$ only on entering state 3, and $\gamma = 0.9$. By backward induction:
$$V^*(s_3) = 0,\quad V^*(s_2) = 1,\quad V^*(s_1) = 0.9,\quad V^*(s_0) = 0.81.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gamma = 0.9
V = np.zeros(4)
V[2] = 1.0                               # reward +1 arriving at s_3
V[1] = gamma * V[2]
V[0] = gamma * V[1]

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.bar(range(4), V, color='steelblue', edgecolor='black')
for i, v in enumerate(V):
    ax.text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=11)
ax.set_xticks(range(4)); ax.set_xticklabels([f'$s_{i}$' for i in range(4)])
ax.set_ylabel('$V^*(s)$'); ax.set_ylim(0, 1.2)
ax.set_title(r'Bellman back-up on 4-state chain ($\gamma=0.9$)')
plt.tight_layout(); plt.show()

---
# Part B · In-Class Exercise

> **Your job**: fill in the `# TODO` blocks in the tasks below. Each task has a collapsed **Solution** cell underneath — try the task yourself first, then expand the solution to compare.
>
> Part B covers exactly **two models**: **DQN** (Tasks B1–B3, plus a tiny training run) and **PPO** (Task B4, plus the canonical clipping visualisation).

In [ ]:
# ── Imports and seeds ────────────────────────────────────────────────────────
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0); random.seed(0)

## Model 1 · Deep Q-Network (DQN)

We will implement DQN on a tiny **4-state chain** MDP so every piece fits in one screen. States are $\{0, 1, 2, 3\}$; actions are `0 = left`, `1 = right`; the terminal reward $+1$ is given on entering state 3; every other step costs $-0.01$; $\gamma = 0.9$.

The environment is already written for you — just read it.

In [ ]:
# ── 4-state chain environment (already written — just read) ─────────────────
class ChainEnv:
    """Tiny 4-state chain: 0 — 1 — 2 — 3 (terminal). Action 0=left, 1=right."""
    def __init__(self, n=4):
        self.n = n
        self.state = 0
    def reset(self):
        self.state = 0
        return self._obs()
    def _obs(self):
        # one-hot state encoding
        x = np.zeros(self.n, dtype=np.float32)
        x[self.state] = 1.0
        return x
    def step(self, a):
        if a == 1:
            self.state = min(self.state + 1, self.n - 1)
        else:
            self.state = max(self.state - 1, 0)
        done = (self.state == self.n - 1)
        reward = 1.0 if done else -0.01
        return self._obs(), reward, done

### Task B1 · Implement `QNetwork`

Fill in the `# TODO` lines so `QNetwork` is a small MLP with two hidden layers of width `hidden=32` and ReLU activations, mapping a state vector of shape `(batch, state_dim)` to a Q-value vector of shape `(batch, n_actions)` — one Q-value per action.

> **Hint.** This is the same 3-linear-layer pattern you used in every earlier week; the only thing specific to DQN is the **output size** — one unit per action, no softmax (Q-values are not probabilities).

In [ ]:
class QNetwork(nn.Module):
    """MLP Q-network: state -> Q-values for each action."""
    def __init__(self, state_dim: int, n_actions: int, hidden: int = 32):
        super().__init__()
        # TODO 1 — build an nn.Sequential with layers:
        #          Linear(state_dim, hidden) -> ReLU
        #       -> Linear(hidden, hidden)    -> ReLU
        #       -> Linear(hidden, n_actions)       (no activation on the output)
        self.net = ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 2 — return Q-values for all actions, shape (batch, n_actions)
        return ...

<details>
<summary><b>▸ Solution · Task B1</b> (click to expand)</summary>

```python
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),   # TODO 1
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, n_actions),              # raw Q-values, no softmax
        )
    def forward(self, x):
        return self.net(x)                             # TODO 2 — (batch, n_actions)
```

**Key points:**
- **Output head has one unit per action**, not one per state. A single forward pass gives every $Q_\theta(s, \cdot)$ in parallel, so the $\max_{a'}$ in the TD target is a cheap `.max(dim=1)`.
- **No activation on the output.** Q-values are unbounded real numbers; applying ReLU/softmax/sigmoid would restrict the representable range and break TD learning.
- **Shapes are the bookkeeping test.** `QNetwork(4, 2)(torch.zeros(5, 4))` must return shape `(5, 2)`.
</details>

### Task B2 · Implement `ReplayBuffer`

Fill in `push` and `sample`. The buffer stores tuples `(s, a, r, s_next, done)`. A random mini-batch of `batch_size` transitions should be returned as a tuple of tensors — all leading dimensions equal to `batch_size`.

> **Hint.** Use `collections.deque(maxlen=capacity)` for FIFO eviction (already done for you). For `sample`, `random.sample` + `zip(*batch)` gives you five parallel tuples that you can pass straight into `torch.as_tensor`.

In [ ]:
class ReplayBuffer:
    """FIFO experience replay for DQN."""
    def __init__(self, capacity: int = 10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        # TODO 1 — append the transition tuple (s, a, r, s_next, done) to self.buffer
        ...

    def sample(self, batch_size: int):
        # TODO 2 — sample `batch_size` transitions uniformly at random,
        #          unpack them into five parallel tuples, and convert to tensors:
        #            states     : FloatTensor (B, state_dim)
        #            actions    : LongTensor  (B,)
        #            rewards    : FloatTensor (B,)
        #            next_states: FloatTensor (B, state_dim)
        #            dones      : FloatTensor (B,)      0.0 if not done, 1.0 if done
        batch = ...
        s, a, r, sn, d = ...
        return (torch.as_tensor(np.array(s), dtype=torch.float32),
                torch.as_tensor(a, dtype=torch.long),
                torch.as_tensor(r, dtype=torch.float32),
                torch.as_tensor(np.array(sn), dtype=torch.float32),
                torch.as_tensor(d, dtype=torch.float32))

    def __len__(self):
        return len(self.buffer)

<details>
<summary><b>▸ Solution · Task B2</b> (click to expand)</summary>

```python
def push(self, s, a, r, s_next, done):
    self.buffer.append((s, a, r, s_next, done))         # TODO 1

def sample(self, batch_size):
    batch      = random.sample(self.buffer, batch_size) # TODO 2a — uniform random
    s, a, r, sn, d = zip(*batch)                         # TODO 2b — parallel tuples
    return (torch.as_tensor(np.array(s),  dtype=torch.float32),
            torch.as_tensor(a,            dtype=torch.long),
            torch.as_tensor(r,            dtype=torch.float32),
            torch.as_tensor(np.array(sn), dtype=torch.float32),
            torch.as_tensor(d,            dtype=torch.float32))
```

**Key points:**
- **Why a buffer at all?** Consecutive transitions from one episode are highly correlated — state $s_{t+1}$ almost always looks like $s_t$. Plain SGD on a stream of such transitions violates its i.i.d. assumption and the network overfits the most recent trajectory. Uniform random sampling from a large buffer **decorrelates** the mini-batch and **reuses** old data (sample efficiency).
- **`deque(maxlen=capacity)` gives FIFO eviction** for free: once full, pushing a new transition drops the oldest one.
- **`done` is stored as a float**, not a bool, because the TD target multiplies by `(1 - done)` — see Task B3.
- **`actions` must be `LongTensor`** so we can `gather` on them without a cast.
</details>

### Task B3 · Implement `dqn_loss`

This is the heart of DQN. Given a mini-batch and a target network, compute:

$$y_i \;=\; r_i \;+\; \gamma \cdot \max_{a'} Q_{\theta^-}(s'_i, a') \cdot (1 - \text{done}_i), \qquad \mathcal{L} \;=\; \frac{1}{B}\sum_i \bigl(y_i - Q_\theta(s_i, a_i)\bigr)^2.$$

Three things to be careful about:

1. **Gather the acted Q-values** — `q_net(s)` has shape `(B, n_actions)`; we want the single value at the action actually taken: `q_net(s).gather(1, a[:, None]).squeeze(1)`.
2. **No gradient through the target.** Wrap the target computation in `torch.no_grad()` (or detach). The target network is frozen *and* we do not want autograd to chase its output.
3. **Terminal transitions must zero the bootstrap.** When `done[i] = 1`, there is no next state to bootstrap from, so $y_i = r_i$.

In [ ]:
def dqn_loss(q_net: nn.Module,
             target_net: nn.Module,
             batch: tuple,
             gamma: float = 0.9) -> torch.Tensor:
    """Compute the mean-squared TD error on a mini-batch."""
    s, a, r, s_next, done = batch

    # TODO 1 — Q-values predicted for the action actually taken:
    #          shape (B,).  Hint: q_net(s).gather(1, a[:, None]).squeeze(1)
    q_sa = ...

    # TODO 2 — TD target y_i = r_i + gamma * max_a' Q_target(s'_i, a') * (1 - done_i)
    #          Wrap in torch.no_grad() so no gradient flows into target_net.
    with torch.no_grad():
        next_q_max = ...                   # shape (B,)
        target     = ...                   # shape (B,)

    # TODO 3 — mean squared error between q_sa and target
    loss = ...
    return loss

<details>
<summary><b>▸ Solution · Task B3</b> (click to expand)</summary>

```python
def dqn_loss(q_net, target_net, batch, gamma=0.9):
    s, a, r, s_next, done = batch
    # TODO 1 — acted Q(s, a)
    q_sa = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)        # (B,)
    # TODO 2 — TD target using the FROZEN target network
    with torch.no_grad():
        next_q_max = target_net(s_next).max(dim=1).values       # (B,)
        target     = r + gamma * next_q_max * (1.0 - done)      # (B,)
    # TODO 3 — mean squared error
    loss = F.mse_loss(q_sa, target)
    return loss
```

**Key points:**
- **`gather(1, a.unsqueeze(1)).squeeze(1)` is the DQN idiom.** It turns the per-action Q matrix `(B, n_actions)` into the per-sample scalar `Q(s, a)`. Memorise it; it will appear on an exam.
- **`torch.no_grad()` on the target is non-optional.** Without it, autograd would back-propagate *through* the target network and its parameters would drift during `.backward()` — the target is supposed to be a fixed reference, not a second learner.
- **The `(1 - done)` mask zeroes bootstrapping at episode end.** At a terminal state there is no next state, so $y_i = r_i$ exactly. Forgetting this is the #1 DQN bug and produces wildly biased Q-values near episode boundaries.
- **Why a *separate* target network and not just `detach()`?** `detach()` stops the gradient, but the values it returns still come from the *current* weights — so the target moves every step. A separate `target_net` that is synced every $C$ steps gives a **stationary target for $C$ steps**, which is the contraction property the theory needs.
</details>

### Tiny DQN training run (already written)

The cell below assembles your `QNetwork`, `ReplayBuffer`, and `dqn_loss` into a complete agent and trains it for **50 episodes** on `ChainEnv`. If your three components are correct, the learned greedy policy at state 0 should be `right` and the estimated $Q^*(s_0, \text{right}) \approx 0.9^2 \approx 0.81$ — matching the backward-induction value from §6 of Part A.

In [ ]:
# ── Tiny 50-episode DQN training loop ───────────────────────────────────────
env         = ChainEnv(n=4)
q_net       = QNetwork(state_dim=4, n_actions=2)
target_net  = QNetwork(state_dim=4, n_actions=2)
target_net.load_state_dict(q_net.state_dict())     # start synchronised
buffer      = ReplayBuffer(capacity=2000)
optimizer   = optim.Adam(q_net.parameters(), lr=5e-3)

GAMMA, BATCH, SYNC_EVERY, EPS = 0.9, 32, 20, 0.2
returns = []

for ep in range(50):
    s, done, ep_ret, t = env.reset(), False, 0.0, 0
    while not done and t < 20:
        if random.random() < EPS:                                 # ε-greedy
            a = random.randint(0, 1)
        else:
            with torch.no_grad():
                a = int(q_net(torch.as_tensor(s).unsqueeze(0)).argmax(1).item())
        s_next, r, done = env.step(a)
        buffer.push(s, a, r, s_next, float(done))
        s = s_next; ep_ret += r; t += 1
        if len(buffer) >= BATCH:                                  # learn step
            loss = dqn_loss(q_net, target_net, buffer.sample(BATCH), gamma=GAMMA)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (ep + 1) % SYNC_EVERY == 0:                                # target sync
        target_net.load_state_dict(q_net.state_dict())
    returns.append(ep_ret)

with torch.no_grad():
    q0 = q_net(torch.eye(4)).numpy()
print(f'Average return over last 10 episodes: {np.mean(returns[-10:]):.3f}')
print(f'Learned Q(s, ·) per state (rows=state, cols=[left, right]):')
print(np.round(q0, 3))
print(f'Greedy action at s_0: {"right" if q0[0, 1] > q0[0, 0] else "left"}')

## Model 2 · Proximal Policy Optimization (PPO)

DQN learns values and acts greedily. **PPO** learns a policy *directly* and bounds how much it changes per update. The single most important piece is the **clipped surrogate objective** from §5:

$$\mathcal{L}^{\text{CLIP}}(\theta) \;=\; \mathbb{E}_t\!\Bigl[\min\!\bigl(r_t(\theta)\,\hat A_t,\; \text{clip}(r_t(\theta),\,1-\varepsilon,\,1+\varepsilon)\,\hat A_t\bigr)\Bigr], \qquad r_t(\theta) = \exp(\log \pi_\theta - \log \pi_{\theta_\text{old}}).$$

Task B4 implements the loss exactly as written. We then plot the canonical two-panel diagram that every PPO paper/blog reproduces.

### Task B4 · Implement `ppo_loss`

Fill in the three `# TODO` lines. The function returns the **negative** mean (so that calling `.backward()` on it corresponds to *maximising* $\mathcal{L}^{\text{CLIP}}$).

> **Hint.**
> - `ratio = torch.exp(log_probs_new - log_probs_old)` — this is numerically stabler than dividing two probabilities.
> - `torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)` — element-wise clip.
> - `torch.min(a, b)` — element-wise min. Then take `.mean()` over the batch.

In [ ]:
def ppo_loss(log_probs_new: torch.Tensor,
             log_probs_old: torch.Tensor,
             advantages:    torch.Tensor,
             clip_eps: float = 0.2) -> torch.Tensor:
    """PPO clipped surrogate objective (returned as a loss to *minimise*)."""
    # TODO 1 — importance ratio r_t(θ) = exp(log π_new - log π_old)
    ratio = ...

    # TODO 2 — clipped surrogate:  min( r * A,  clip(r, 1-ε, 1+ε) * A )
    unclipped = ...
    clipped   = ...
    surrogate = ...

    # TODO 3 — PPO *maximises* the surrogate, so the loss is its negative mean
    loss = ...
    return loss

<details>
<summary><b>▸ Solution · Task B4</b> (click to expand)</summary>

```python
def ppo_loss(log_probs_new, log_probs_old, advantages, clip_eps=0.2):
    # TODO 1 — importance ratio in log-space (stable)
    ratio     = torch.exp(log_probs_new - log_probs_old)
    # TODO 2 — clipped surrogate
    unclipped = ratio * advantages
    clipped   = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
    surrogate = torch.min(unclipped, clipped)
    # TODO 3 — negate and average
    loss      = -surrogate.mean()
    return loss
```

**Key points:**
- **Why log-space for the ratio?** Policy probabilities can be tiny (think $10^{-6}$ for one token out of a 50 000-vocab softmax); a literal division blows up numerically. `exp(log_new - log_old)` stays in the well-conditioned regime.
- **Why `min`, not `clip` alone?** If you only clipped the ratio, then for $\hat A > 0$ an exploding ratio capped at $1+\varepsilon$ still produces a positive objective to maximise — nothing stops the update. The `min` ensures that once the ratio leaves the trust region on the *favourable* side, the gradient is zero. Outside the trust region PPO is **pessimistic**, which is exactly the "trust" part of trust-region.
- **Sign convention.** We return $-\mathcal{L}^{\text{CLIP}}$ because the PyTorch convention is to **minimise** a loss with `.backward()`. A positive advantage + ratio above 1 gives a *more negative* loss, which is good.
- **Advantages should be normalised** (subtract mean, divide by std) per batch before calling `ppo_loss`. This is a standard variance-reduction trick and is done *outside* the loss function.
</details>

### Canonical PPO visualisation

Use `ppo_loss` to reproduce the two-panel diagram. We sweep the importance ratio $r$ from 0 to 2.5, fix a single synthetic advantage, and plot **one point per `ppo_loss` call** — the left panel for $\hat A = +1$, the right for $\hat A = -1$. The dashed grey / orange curves are the *unclipped* and *clipped-only* components, for reference.

In [ ]:
# ── PPO clipping landscape: evaluate ppo_loss at each ratio point ──────────
clip_eps   = 0.2
ratios     = torch.linspace(0.0, 2.5, 500)
log_old    = torch.zeros_like(ratios)          # old log-prob = 0
log_new    = torch.log(ratios + 1e-8)          # so that exp(new - old) = ratio

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, A_val, title, colour in [
    (axes[0], +1.0, r'$\hat A_t > 0$  (good action)',  'steelblue'),
    (axes[1], -1.0, r'$\hat A_t < 0$  (bad action)',   'coral'),
]:
    A = torch.full_like(ratios, A_val)
    unclipped = (ratios * A).numpy()
    clipped   = (torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * A).numpy()
    # PPO objective = min(unclipped, clipped); our ppo_loss returns its NEGATIVE mean,
    # so to get the objective for plotting we evaluate per-point and flip the sign.
    objective = np.array([-ppo_loss(torch.tensor([ln]),
                                    torch.tensor([0.0]),
                                    torch.tensor([A_val]),
                                    clip_eps).item()
                          for ln in log_new])
    ax.plot(ratios, unclipped, '--', color='gray',   alpha=0.6, label='unclipped $r\\cdot\\hat A$')
    ax.plot(ratios, clipped,   '--', color='orange', alpha=0.6, label='clip-only')
    ax.plot(ratios, objective, color=colour, lw=2.5, label='PPO objective (min)')
    ax.axvline(1 - clip_eps, color='red', ls=':', alpha=0.5)
    ax.axvline(1 + clip_eps, color='red', ls=':', alpha=0.5)
    ax.set_xlabel(r'importance ratio $r_t(\theta)$')
    ax.set_ylabel('objective')
    ax.set_title(title); ax.legend(fontsize=9)

plt.suptitle(r'PPO clipped surrogate  ($\varepsilon = 0.2$)', fontweight='bold')
plt.tight_layout(); plt.show()

**What the plot shows (memorise this):**
- **$\hat A_t > 0$** (left): the objective **rises linearly** with the ratio until $r = 1 + \varepsilon$, then **flattens** — the gradient is zero beyond the trust region, so the update stops pushing $\pi_\theta$ further. The agent is rewarded for increasing the probability of a good action, but only up to a capped amount per epoch.
- **$\hat A_t < 0$** (right): the objective **falls linearly** with the ratio until $r = 1 - \varepsilon$, then **flattens** — the gradient is zero once the bad action's probability has been sufficiently suppressed. The agent cannot suppress a bad action to zero in a single PPO update.
- **The flat region is where `min` does its job**: without it, the dashed clip-only curve (orange) would still produce gradient signal outside the trust region and the update could run away.

---
# Part C · Exam-Style Questions

> Three short-answer questions at medium-high difficulty. Each has a hidden **Model Answer** cell directly below with the key points the tutor will draw out during discussion.

## Q1 · Why DQN needs a target network

Suppose you implement DQN exactly as in Tasks B1–B3, but you **remove the target network** — that is, the TD target uses the *same* `q_net` that is being optimised:
$$y_i \;=\; r_i \;+\; \gamma \, \max_{a'} Q_\theta(s'_i, a') \, (1 - \text{done}_i).$$

**(a)** Describe the **forward-pass symptom** you would observe on a training run: what happens to the TD loss curve over the first few thousand steps, and why?

**(b)** Describe the **gradient symptom**: what does the gradient $\nabla_\theta \mathcal{L}$ pick up that it would *not* pick up with a frozen target, and why does this break the contraction argument that guarantees convergence to $Q^*$?

**(c)** Someone suggests that a single `.detach()` on `Q_\theta(s', a')` is enough — no need for a separate target network, no need to sync weights every $C$ steps. In one or two sentences, explain why this is only a partial fix and what is still unstable.

<details>
<summary><b>▸ Model Answer · Q1</b></summary>

**(a) Forward-pass symptom.** The TD loss curve typically **drops rapidly** for the first few hundred steps and then **oscillates wildly** or **diverges outright**. The reason: every gradient step moves $Q_\theta(s, a)$ *towards* the target $y = r + \gamma \max_{a'} Q_\theta(s', a')$, but it *also* moves the target itself, because the target is computed with the same network. You are chasing a point that moves in response to your own step — a classic **moving-target** problem. In practice the Bellman residual does not settle; you see large fluctuations and, on harder environments, NaN explosions.

**(b) Gradient symptom.** With a frozen target, the gradient is
$$\nabla_\theta \mathcal{L} \;=\; -\bigl(y - Q_\theta(s, a)\bigr)\, \nabla_\theta Q_\theta(s, a),$$
the **semi-gradient** — only the prediction depends on $\theta$. Removing the target makes $y$ itself a function of $\theta$, so the full gradient gains an extra term
$$-\bigl(y - Q_\theta(s, a)\bigr)\,\gamma\, \nabla_\theta \bigl[\max_{a'} Q_\theta(s', a')\bigr]$$
that pulls the target *toward* the prediction on every step. The Bellman operator is a $\gamma$-contraction *only when the target is held fixed*; once we let both endpoints move, the iteration is no longer guaranteed to be a contraction, and Q-learning's convergence theorem does not apply. Empirically the result is oscillation or divergence.

**(c) Is `.detach()` enough?** `.detach()` stops the gradient from flowing into $\theta$ through the target, which fixes the *gradient* symptom. But the *values* returned by the detached expression still come from the **current** $\theta$, so the target moves every single step — only the semi-gradient property is restored, not the stationary-target property. A separate `target_net` synced every $C$ steps gives a **stationary target for $C$ steps**, which is what the contraction argument actually needs in the function-approximation setting. `.detach()` = half the fix; the target network is the other half.

**Key points the tutor will land:**
- Moving-target $\Rightarrow$ broken contraction $\Rightarrow$ loss diverges.
- Semi-gradient vs. full gradient — name the distinction.
- `.detach()` fixes the gradient but not the stationarity; you need *both*.
</details>

## Q2 · Reading the PPO clipping landscape

Consider the PPO clipped surrogate with $\varepsilon = 0.2$, evaluated for a single transition with advantage $\hat A_t$ and importance ratio $r_t = r$.

**(a)** For a **positive** advantage $\hat A_t = +1$, write $\mathcal{L}^{\text{CLIP}}$ as an explicit piecewise function of $r$, and state the value of $\partial \mathcal{L}^{\text{CLIP}} / \partial r$ in each piece. Sketch the curve in words (three regions).

**(b)** Repeat for $\hat A_t = -1$. In which region is the gradient **zero**, and what is the practical interpretation of that region?

**(c)** A student proposes to *drop the outer `min`* and just use the clipped term $\text{clip}(r, 0.8, 1.2) \cdot \hat A_t$ as the loss. Give one concrete failure case — a scenario where this modified loss produces a **wrong-direction** or **runaway** update that the original PPO objective correctly suppresses.

<details>
<summary><b>▸ Model Answer · Q2</b></summary>

**(a) Positive advantage, $\hat A_t = +1$.** Here $\text{clip}(r, 0.8, 1.2) \cdot 1 = \text{clip}(r, 0.8, 1.2)$ and the `min` picks the smaller of $r$ and the clipped value. For $r < 1$ the clipped term is at least $0.8 > r$ is *not* always true — let's just do each region cleanly:

$$\mathcal{L}^{\text{CLIP}}(r) \;=\; \begin{cases} r, & 0 \le r \le 1.2, \\[2pt] 1.2, & r > 1.2. \end{cases}$$

(Note: for $r < 0.8$, $\min(r, 0.8) = r$; for $0.8 \le r \le 1.2$, $\min(r, r) = r$; for $r > 1.2$, $\min(r, 1.2) = 1.2$.)

So
$$\frac{\partial \mathcal{L}^{\text{CLIP}}}{\partial r} \;=\; \begin{cases} 1, & r < 1.2, \\[2pt] 0, & r > 1.2. \end{cases}$$

**Three regions in words.** (i) Well below the trust region ($r < 0.8$): the objective is still $r$, gradient $+1$, the update still pushes the probability *up* — PPO does not "repel" you back into the trust region, it just doesn't penalise you for being below. (ii) Inside the trust region ($0.8 \le r \le 1.2$): linear with slope $+1$, normal policy-gradient behaviour. (iii) Above the trust region ($r > 1.2$): **flat**, gradient $0$, no further reward for increasing the probability — the update *halts* on this sample.

**(b) Negative advantage, $\hat A_t = -1$.** Symmetrically,
$$\mathcal{L}^{\text{CLIP}}(r) \;=\; \begin{cases} -0.8, & r < 0.8, \\[2pt] -r, & 0.8 \le r \le 2.5. \end{cases}$$

The **gradient is zero for $r < 0.8$**. Practical interpretation: once we have already reduced the probability of a bad action to 80 % of its old value, PPO **stops pushing it further down on this sample**. Without the flat region, a single large negative advantage could drive a probability to zero in one epoch — exactly the destructively-large update the clip is designed to prevent.

**(c) Dropping the outer `min`.** Consider $\hat A_t = +1$ and $r = 5$ (the new policy has, for whatever reason, put 5× as much probability on this action as the old one). With the original objective, $\mathcal{L}^{\text{CLIP}} = \min(5, 1.2) = 1.2$ — gradient zero, *no further update*. With the modified "clip-only" loss, the value is $\text{clip}(5, 0.8, 1.2) = 1.2$ — also flat, in this case. So for $\hat A > 0$ the clip-only variant happens to coincide above the trust region.

The failure case is on the **other side**: $\hat A_t = -1$ and $r = 5$ (a formerly rare-but-bad action has become dominant — exactly the case we most urgently need to correct). Original PPO gives $\min(5 \cdot -1,\; 1.2 \cdot -1) = \min(-5, -1.2) = -5$ — a *large negative objective*, and maximising means pushing $r$ **down** strongly. The clip-only version gives $1.2 \cdot -1 = -1.2$ with zero gradient — **no corrective force at all**, even though the action is bad and its probability has ballooned. The `min` is what ensures PPO stays **pessimistic** on the "wrong" side of the trust region; without it, catastrophic probability inflations go uncorrected.

**Key points the tutor will land:**
- The clip gives the *shape*; the `min` gives the *direction*.
- PPO is **not** a symmetric trust region — it is pessimistic outside the box.
- Zero-gradient regions are a feature, not a bug: they bound the per-sample update.
</details>

## Q3 · On-policy vs. off-policy: why replay buffers do not transplant

DQN stores transitions in a replay buffer and reuses them for many gradient updates. PPO collects trajectories under $\pi_{\theta_\text{old}}$, performs a few epochs of updates, and then **throws the data away** and collects new trajectories.

**(a)** Explain, in terms of the Bellman equation, why DQN is an **off-policy** algorithm and is therefore free to reuse old transitions — even transitions collected by a very different policy.

**(b)** Explain why PPO is an **on-policy** algorithm and why re-using data from a stale policy *would* bias its gradient. In which quantity of the PPO objective does the staleness show up, and what does the clipping mechanism do to bound (but not eliminate) the bias?

**(c)** A student suggests: "Why not just apply DQN-style clipping to the PPO loss — clip the TD error to $\pm\varepsilon$ and put PPO's old transitions in a replay buffer for free sample efficiency?" Explain the conceptual error. (Hint: what does the `min`/`clip` in PPO *bound* that has no analogue in the DQN loss, and what would clipping the TD error in DQN actually accomplish?)

<details>
<summary><b>▸ Model Answer · Q3</b></summary>

**(a) Why DQN is off-policy.** The Bellman optimality equation
$$Q^*(s, a) = \mathbb{E}_{s' \sim P}\!\bigl[r + \gamma \max_{a'} Q^*(s', a')\bigr]$$
is a statement about the **environment dynamics** $P$ and the **greedy** operator $\max_{a'}$ — the behaviour policy that *generated* the transition $(s, a, r, s')$ does not appear anywhere on the right-hand side. As long as the transition tuple is a valid draw from $P$ (which it always is, because it actually happened in the environment), it is a valid Monte-Carlo sample of the Bellman target regardless of which policy chose $a$. This is exactly why DQN can reuse millions of old transitions from a replay buffer, and also why $\varepsilon$-greedy exploration is allowed — the behaviour policy and the target policy (greedy) are formally different, but Q-learning's update is agnostic.

**(b) Why PPO is on-policy.** PPO's gradient estimator is built from the identity
$$\nabla_\theta J(\theta) \;=\; \mathbb{E}_{a \sim \pi_{\theta_\text{old}}}\!\Bigl[\frac{\pi_\theta(a \mid s)}{\pi_{\theta_\text{old}}(a \mid s)} \, \nabla_\theta \log \pi_\theta(a \mid s) \cdot \hat A_t\Bigr],$$
which is an **importance-sampling** estimator centred on $\pi_{\theta_\text{old}}$. It is only unbiased when the expectation is taken over samples drawn from **the same distribution** the importance weight was computed against. As $\theta$ drifts from $\theta_\text{old}$, the ratio $r_t(\theta)$ deviates from 1, the true expectation is no longer what the batch approximates, and the estimator becomes **biased**. The staleness shows up in the importance ratio. The clip $\text{clip}(r_t, 1-\varepsilon, 1+\varepsilon)$ prevents the ratio from growing unboundedly — it **bounds** the bias-amplification per sample — but it does **not** restore unbiasedness. This is why PPO re-collects fresh trajectories every few epochs: it trusts the importance weight only inside the clip box.

**(c) The conceptual error.** The `min`/`clip` in PPO is not a "TD-error clipper" — it is a **trust-region bound on the policy change**. It clips the *importance ratio* $\pi_\theta / \pi_{\theta_\text{old}}$, a quantity that has no analogue in DQN, because DQN has no $\pi_\theta$ to ratio against a $\pi_{\theta_\text{old}}$ in the first place (DQN is implicitly policy-free — it learns $Q$ and acts greedily). Clipping the TD error in DQN would be a completely different operation: it would just be a form of **Huber loss / gradient clipping** on the Bellman residual, bounding the magnitude of the regression target per sample. That is a mild variance-reduction trick and is in fact sometimes done in practice (Rainbow DQN uses Huber loss) — but it does **not** make PPO's ratio-clipping and DQN's error-clipping the same operation, and it does **not** justify dumping old-policy trajectories into a PPO replay buffer. DQN can reuse old data because the Bellman target is policy-agnostic; PPO cannot because the policy gradient is not. No amount of loss-side clipping changes that.

**Key points the tutor will land:**
- The Bellman target has no $\pi_\theta$ in it — that is what "off-policy" *means*.
- The policy gradient has $\pi_\theta$ front and centre, and PPO is an importance-sampled version of it.
- Clipping in the two algorithms bounds **different things**: ratio (policy change) vs. TD residual (regression magnitude).
- Replay buffer + PPO = biased gradient; you would need a different correction (e.g. V-trace, Retrace) to make it work.
</details>

---
## Summary

| Section | Key concept |
|---|---|
| **§1–§2** | MDP tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$; Bellman optimality equation $Q^*(s,a) = \mathbb{E}[r + \gamma \max_{a'} Q^*(s', a')]$; tabular TD update |
| **§3** | DQN = neural $Q_\theta$ + **replay buffer** (decorrelate samples, reuse data) + **target network** (stationary TD target for $C$ steps) |
| **§4–§5** | Policy gradient $\nabla J = \mathbb{E}[\nabla \log \pi_\theta \cdot \hat A]$; REINFORCE has high variance; PPO clipped surrogate $\mathcal{L}^{\text{CLIP}} = \mathbb{E}[\min(r\hat A,\, \text{clip}(r, 1{\pm}\varepsilon)\hat A)]$ bounds per-step change |
| **B1–B3** | `QNetwork`, `ReplayBuffer`, `dqn_loss` → trained on a 4-state chain in 50 episodes |
| **B4** | `ppo_loss` with the log-space ratio; canonical two-panel clipping landscape for $\hat A > 0$ and $\hat A < 0$ |

**Take-aways:**
- **Value-based (DQN)** is off-policy — old data is fair game, but the target network and replay buffer are *both* needed for stability.
- **Policy-gradient (PPO)** is on-policy — the clip bounds how stale we let the data get before re-collecting.
- The LLM-alignment family (RLHF, DPO, SLiC-HF, GRPO) is built on top of PPO's clipped objective — see the self-study notebook.